# Full training run (run only after the PoC decision gate passes)

<a href="https://colab.research.google.com/github/saranunt-0/layer-mapping/blob/main/notebooks/02_train_full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Same code as the PoC, scaled up, with everything kept on Google Drive so a disconnect only costs the last
`save_every` steps. **Re-running the notebook resumes automatically.**

| Setting | Default | Notes |
|---|---|---|
| Runtime | A100 (or L4) | T4 works with `BATCH = 8` but takes about 3× longer |
| Synthetic images | 30,000 | ≈4 GB; generated once, then cached on Drive as a tar |
| Steps × batch | 30,000 × 16 | ≈480k samples ≈ 16 epochs |
| Wall-clock | measure it | the PoC log prints `s/step`; hours ≈ s/step × steps / 3600 |

Drive needs about 5 GB free (dataset cache + checkpoints).

## 0. Setup

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/saranunt-0/layer-mapping.git"
BRANCH = "main"
REPO_DIR = "/content/layer-mapping"

if not os.path.exists(REPO_DIR):
    url = REPO_URL
    try:
        # Only needed while the repo is private: add a GITHUB_TOKEN secret (key icon, left sidebar).
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
sys.path.insert(0, REPO_DIR)

import torch, transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.45"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers"], check=True)
    raise SystemExit("transformers was upgraded: Runtime > Restart session, then run all cells again.")
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - Runtime > Change runtime type > GPU")
print("CPU cores:", os.cpu_count())

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json, time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from layermap import data, layers, metrics, synth
from layermap import model as lm
from layermap import train as lt


def load_rgb(path, max_side=1024):
    rgb = cv2.imread(str(path), cv2.IMREAD_COLOR)[..., ::-1]
    s = max_side / max(rgb.shape[:2])
    if s < 1:
        rgb = cv2.resize(rgb, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    return np.ascontiguousarray(rgb)


def show_row(images, titles, size=4.0):
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size))
    for ax, im, t in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(im)
        ax.set_title(t, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def mean_metrics(rows):
    return {k: round(float(np.nanmean([r[k] for r in rows])), 4) for k in rows[0]}


def prior_metrics(ds):
    """Score of the "lower in the frame = closer" rule on an eval dataset."""
    return mean_metrics([metrics.all_metrics(metrics.vertical_prior(ds[i][1].shape), ds[i][1])
                         for i in range(len(ds))])

## 1. Configuration

In [ ]:
DRIVE = Path("/content/drive/MyDrive/layer-mapping")
RUN_DIR = DRIVE / "runs" / "full_v1"          # change the name to start a fresh run
N_TRAIN, N_VAL = 30_000, 500
SIZE = 518
BATCH = 16                                    # A100 40 GB: 16; L4 / T4: 8
STEPS = 30_000
EXTRA_TRAIN_DIRS = []                         # e.g. [str(DRIVE / "real_train")]: your labelled paintings
REAL_EVAL_DIR = DRIVE / "real_eval"           # images/ + labels/, never trained on
DRIVE.mkdir(parents=True, exist_ok=True)

## 2. Data (generated once, cached on Drive)

Writing 30k small files straight to Drive is slow, so data is generated on the local disk and copied to Drive as
a single tar. Later sessions just extract the tar. The cache name includes the generator version.

In [ ]:
LOCAL = Path("/content/data")
LOCAL.mkdir(exist_ok=True)
cfg_synth = synth.SynthConfig(size=576)


def cached_split(name, n, start_seed):
    out = LOCAL / name
    tar = DRIVE / "cache" / f"synth_v{synth.GENERATOR_VERSION}_{name}_{n}_s{start_seed}.tar"
    have = len(list((out / "labels").glob("*.png"))) if (out / "labels").exists() else 0
    if have < n and tar.exists():
        print("extracting", tar.name)
        subprocess.run(["tar", "-xf", str(tar), "-C", str(LOCAL)], check=True)
    t = time.time()
    data.generate_dataset(out, n, start_seed=start_seed, cfg=cfg_synth)  # only renders what is missing
    print(f"{name}: ready in {time.time() - t:.0f}s")
    if not tar.exists():
        tar.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["tar", "-cf", str(tar), "-C", str(LOCAL), name], check=True)
    return out


train_dir = cached_split("train", N_TRAIN, 0)
val_dir = cached_split("val", N_VAL, 1_000_000)

## 3. Train (resumes from `RUN_DIR/last` if it exists)

In [ ]:
cfg_train = lt.TrainConfig(
    train_dirs=[str(train_dir)] + EXTRA_TRAIN_DIRS, val_dir=str(val_dir), out_dir=str(RUN_DIR),
    size=SIZE, batch_size=BATCH, steps=STEPS, warmup=500, lr_backbone=5e-6, lr_head=5e-5,
    eval_every=2000, save_every=1000, log_every=50, num_workers=min(8, os.cpu_count()),
)
history = lt.train(cfg_train)

In [ ]:
hist = [json.loads(l) for l in (RUN_DIR / "history.jsonl").read_text().splitlines()]
tr = [h for h in hist if "loss" in h]
va = [h for h in hist if "val_score" in h]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([h["step"] for h in tr], [h["loss"] for h in tr]); ax[0].set_title("train loss")
for k in ("val_acc_all", "val_acc_local", "val_acc_hard"):
    ax[1].plot([h["step"] for h in va], [h[k] for h in va], label=k)
ax[1].legend(); ax[1].set_title("validation (synthetic)")
plt.show()

## 4. Evaluate: vertical prior vs zero-shot vs fine-tuned

In [ ]:
base = lm.load_model(lm.DEFAULT_BACKBONE)
ft = lm.load_model(str(RUN_DIR / "best"))
sets = {"synthetic val": data.LayerFolderDataset(val_dir, SIZE, train=False)}
if (REAL_EVAL_DIR / "labels").exists():
    sets["real paintings"] = data.LayerFolderDataset(REAL_EVAL_DIR, SIZE, train=False)
for name, ds in sets.items():
    print(f"== {name} ({len(ds)} images)")
    for label, res in [("vertical prior", prior_metrics(ds)), ("zero-shot", lt.evaluate(base, ds)),
                       ("fine-tuned", lt.evaluate(ft, ds))]:
        print(f"  {label:15s}", {m: round(x, 4) for m, x in res.items()})

## 5. Batch export: layers for a folder of paintings

In [ ]:
SRC = DRIVE / "paintings"          # put your paintings here
DST = DRIVE / "exports" / RUN_DIR.name
N_LAYERS, INFER_SIZE = 6, 770

for p in (sorted(SRC.glob("*")) if SRC.exists() else []):
    if p.suffix.lower() not in (".jpg", ".jpeg", ".png", ".webp"):
        continue
    rgb = load_rgb(p, max_side=2048)
    labels = layers.depth_to_layers(lm.predict(ft, rgb, size=INFER_SIZE), N_LAYERS, rgb=rgb)
    layers.export_layers(rgb, labels, DST / p.stem)
    print("exported", p.name)

## 6. Releasing the model (optional)

`RUN_DIR / "best"` is a standard Hugging Face model folder (`config.json` + `model.safetensors`).
Before publishing it as open source, check that **every** training input is cleared for it:

* backbone: Depth Anything V2 **Small** (Apache-2.0); keep its license and attribution
* synthetic data: generated by this repo
* any `EXTRA_TRAIN_DIRS`: only public-domain / CC0 paintings with labels you drew yourself
* **no** Adobe Illustrator's Depth outputs, MMSVG, WikiArt, or other non-commercial data

```python
# from huggingface_hub import login; login()
# ft.push_to_hub("your-name/layermap-v1", private=True)
```